
# Feature Selection Pipeline: Subset Smoke Test

This notebook exercises feature selection using **both**:
- `selection_functions.py` (direct function call)
- `selection_script.py` (CLI pipeline)

It mirrors your script paths:
- `data/transcriptomics_filtered.joblib` (predictors **X**)
- `data/proteomics_matched.joblib` (responses **Y**, proteins)

It lets you choose a small, controllable subset of samples/genes to keep runtime fast, then runs:
1. `feature_selection(...)` (ANOVA, Pearson, Spearman with percentile threshold)
2. `naive_cluster_feature_selection(...)` (cluster-based top-\*N\* Pearson)
3. The CLI script: `!python selection_script.py ...`
4. A batch loop cell to run feature selection for many proteins and save summaries.

> **Note on clusters:** `selection_functions.py` loads `protein_clusters_k3.csv` at import time. Make sure that CSV is in the **same folder** as the notebook and the `.py` files when you run it.


In [1]:

# --- Controls you can tweak ---
N_SAMPLES = None         # Set to None to keep all samples
N_GENES = None          # Set to None to keep all genes
PERCENTILE = 5.0        # For feature_selection()
RANDOM_SEED = 42

#force a specific protein; set to None to auto-pick a good one
FORCE_PROTEIN_NAME = 'A2M'

# Subset strategy for genes: "random" or "variance"
GENE_SUBSET_STRATEGY = "random"


In [2]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
from joblib import load, dump
import subprocess, shlex
import matplotlib.pyplot as plt
import importlib
from time import perf_counter

from collections import defaultdict
import warnings
warnings.filterwarnings("ignore", message=".*correlation coefficient is not defined.*")

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
TRANSCRIPTOMICS_FILE = DATA_DIR / "transcriptomics_filtered.joblib"  # X
PROTEOMICS_FILE = DATA_DIR / "proteomics_matched.joblib"                 # Y

# Make sure we can import local modules
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")
print(f"Data dir:     {DATA_DIR}")


Project root: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection
Data dir:     c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection\data


In [3]:
try:
    sf = importlib.import_module("selection_functions")
    _ = getattr(sf, "feature_selection")
    _ = getattr(sf, "naive_cluster_feature_selection_indices")
    print("Imported selection_functions.py successfully.")
    try:
        cl = getattr(sf, "clusters")
        print(f"clusters loaded with {len(cl)} entries.")
    except Exception as e:
        print("Warning: 'clusters' not available on import:", e)
except Exception as e:
    raise ImportError(
        "Could not import selection_functions.py. Ensure this notebook sits next to your .py files "
        "and that protein_clusters_k3.csv is available."
    ) from e


Imported selection_functions.py successfully.
clusters loaded with 5113 entries.


In [4]:

print("Loading matrices...")
X = load(TRANSCRIPTOMICS_FILE)   # samples x genes
Y = load(PROTEOMICS_FILE)        # samples x proteins

X.index.name = "samples"
Y.index.name = "samples"
X.columns.name = "genes"
Y.columns.name = "proteins"

print("Loaded:")
print(f"  X (transcriptomics): {X.shape}")
print(f"  Y (proteomics):      {Y.shape}")

common_samples = X.index.intersection(Y.index)
if len(common_samples) == 0:
    raise ValueError("No overlapping samples between X and Y.")
X = X.loc[common_samples]
Y = Y.loc[common_samples]
print(f"Common samples: {len(common_samples)}")


Loading matrices...
Loaded:
  X (transcriptomics): (369, 22614)
  Y (proteomics):      (369, 5113)
Common samples: 369


In [5]:

np.random.seed(RANDOM_SEED)

if N_SAMPLES is None:
    chosen_samples = X.index
else:
    n = min(int(N_SAMPLES), X.shape[0])
    chosen_samples = pd.Index(np.random.choice(X.index, size=n, replace=False))

X_sub = X.loc[chosen_samples]
Y_sub = Y.loc[chosen_samples]

if N_GENES is None or N_GENES >= X_sub.shape[1]:
    chosen_genes = X_sub.columns
else:
    if GENE_SUBSET_STRATEGY == "variance":
        var = X_sub.var(axis=0, skipna=True)
        chosen_genes = var.sort_values(ascending=False).head(int(N_GENES)).index
    else:
        chosen_genes = pd.Index(np.random.choice(X_sub.columns, size=int(N_GENES), replace=False))

X_sub = X_sub.loc[:, chosen_genes]

num_cols_before = X_sub.shape[1]
X_sub = X_sub.dropna(axis=1, how="any")
num_cols_after = X_sub.shape[1]
if num_cols_after < num_cols_before:
    print(f"Dropped {num_cols_before - num_cols_after} gene(s) due to NaNs (subset).")

print("Subset shapes:")
print(f"  X_sub: {X_sub.shape}")
print(f"  Y_sub: {Y_sub.shape}")


Subset shapes:
  X_sub: (369, 22614)
  Y_sub: (369, 5113)


In [6]:

if FORCE_PROTEIN_NAME is not None:
    protein_name = FORCE_PROTEIN_NAME
    if protein_name not in Y_sub.columns:
        raise ValueError(f"Requested protein '{protein_name}' not found in proteomics subset.")
else:
    nonnull_counts = Y_sub.notna().sum().sort_values(ascending=False)
    protein_name = nonnull_counts.index[0]

Y_sub = Y_sub.loc[Y_sub[protein_name].notna()]
X_sub = X_sub.loc[Y_sub.index]

print(f"""Protein selected: {protein_name}
Non-null values for {protein_name}: {Y_sub[protein_name].notna().sum()}""")


Protein selected: A2M
Non-null values for A2M: 369


### All correlations computed

In [7]:
print("Running feature_selection (ANOVA)")
t0 = perf_counter()
fs_df = sf.feature_selection_anova_only(
    X=X_sub,
    y=Y_sub,
    protein_name=protein_name,
    percentile=PERCENTILE,
    n_samples=None
)
t1 = perf_counter()
print(f"Done in {t1 - t0:.2f}s")
fs_df.head()


Running feature_selection (ANOVA)
Done in 59.18s


,anova
LOO_sample,
MDAMB468_BREAST,"{'A1BG': 24.16378843365151, 'A2M': 59.88947417..."
SH4_SKIN,"{'A1BG': 23.492211426591226, 'A2M': 56.6318785..."
AU565_BREAST,"{'A1BG': 24.142535343521413, 'A2M': 59.7112884..."
KMRC1_KIDNEY,"{'A1BG': 25.550137834690144, 'A2M': 60.4892399..."
CAL51_BREAST,"{'A1BG': 24.007377500898073, 'A2M': 59.5760843..."


In [8]:

def _count_features(cell):
    if isinstance(cell, dict):
        return len(cell)
    return 0

counts = fs_df.applymap(_count_features)
summary = counts.agg(['mean','median','min','max']).T
print("Selected-feature counts per LOO (summary):")
display(summary)

example_sample = counts.index[0]
def _top_k(d, k=10):
    if not isinstance(d, dict) or len(d)==0:
        return []
    return sorted(d.items(), key=lambda kv: abs(kv[1]), reverse=True)[:k]

example = {
    'anova_top10': _top_k(fs_df.loc[example_sample, 'anova']),
    'pearson_top10': _top_k(fs_df.loc[example_sample, 'pearson']),
    'spearman_top10': _top_k(fs_df.loc[example_sample, 'spearman']),
}
example


Selected-feature counts per LOO (summary):


C:\Users\hp\AppData\Local\Temp\ipykernel_44452\2306689009.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  counts = fs_df.applymap(_count_features)


,mean,median,min,max
anova,1131.0,1131.0,1131.0,1131.0


KeyError: 'pearson'

### Only Pearson and using Clusters

In [ ]:

print("Running naive_cluster_feature_selection (cluster-based top-N Pearson)...")
t0 = perf_counter()
naive_df = sf.naive_cluster_feature_selection(
    X=X_sub,
    y=Y_sub,
    protein_name=protein_name
)
t1 = perf_counter()
print(f"Done in {t1 - t0:.2f}s")
naive_df.head()


Running naive_cluster_feature_selection (cluster-based top-N Pearson)...


Done in 449.38s


,pearson
sample,
HDMYZ_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,"[RBMS3-AS3, TNS2, TBC1D9, COL27A1, IFT22, ARHG..."
JHH5_LIVER,"[RBMS3-AS3, TBC1D9, TNS2, COL27A1, IFT22, APMA..."
CAOV3_OVARY,"[RBMS3-AS3, TBC1D9, TNS2, COL27A1, IFT22, APMA..."
K562_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,"[RBMS3-AS3, TNS2, TBC1D9, COL27A1, SNX33, IFT2..."
NCIH1693_LUNG,"[COL27A1, TBC1D9, TNS2, RBMS3-AS3, APMAP, IFT2..."


In [ ]:

naive_counts = naive_df['pearson'].map(lambda lst: len(lst) if isinstance(lst, list) else 0)
print("Naive selection counts (first 10):")
display(naive_counts.head(10))



Naive selection counts (first 10):


sample
HDMYZ_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE    1000
JHH5_LIVER                                  1000
CAOV3_OVARY                                 1000
K562_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE     1000
NCIH1693_LUNG                               1000
DANG_PANCREAS                               1000
OE33_OESOPHAGUS                             1000
NCIH2126_LUNG                               1000
OV90_OVARY                                  1000
LOXIMVI_SKIN                                1000
Name: pearson, dtype: int64

Stats: mean=1000.0, median=1000.0, min=1000, max=1000


In [ ]:

OUT_DIR = DATA_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

fs_path = OUT_DIR / f"{protein_name}_feature_selection_subset.joblib"
naive_path = OUT_DIR / f"{protein_name}_naive_feature_selection_subset.joblib"

dump(fs_df, fs_path)
dump(naive_df, naive_path)

print("Saved:")
print("  ", fs_path)
print("  ", naive_path)


Saved:
   c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection\data\A2M_feature_selection_subset.joblib
   c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection\data\A2M_naive_feature_selection_subset.joblib



## Run the CLI pipeline (`selection_script.py`)

This cell runs your script end-to-end on the currently chosen `protein_name`, saving the **full** (not subset) LOO results into `data/{protein}_feature_selection_LOO.joblib` as in your script.


In [9]:


cmd = f'python selection_script.py --protein "{protein_name}" --percentile {PERCENTILE}'
print("Running:", cmd)
proc = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print("--- STDERR ---")
    print(proc.stderr)


Running: python selection_script.py --protein "A2M" --percentile 5.0
Loading data...
Running naive feature selection (cluster-aware) for 'A2M'...
Saved compact indices to: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection\data\A2M_feature_selection_LOO_indices.npz
Wrote counts CSV: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. elasticNet\feature_selection\data\A2M_feature_selection_counts.csv
Done.



## Batch loop: run `feature_selection(...)` for many proteins

Configure either:
- `PROTEIN_LOOP_LIST` — list of specific protein names, **or**
- `PROTEIN_LOOP_TOP_K` — automatically pick the top-K proteins by non-null count.

For each protein, we align samples (drop NaNs for that protein), run `feature_selection(...)` on the **current subset `X_sub`**, and save a Joblib file plus a small CSV summary of selected-feature counts per LOO split.


In [ ]:

# --- Batch controls ---
PROTEIN_LOOP_LIST = ['A2M']         # e.g., ["TP53", "EGFR", "MYC"]; set to None to auto-pick top-K
PROTEIN_LOOP_TOP_K = 5           # used only if PROTEIN_LOOP_LIST is None
RUN_NAIVE_IN_LOOP = False        # set True to also run naive_cluster_feature_selection in the loop
BATCH_PERCENTILE = PERCENTILE    # can override here if desired


In [45]:

def _count_dict_features(cell):
    return len(cell) if isinstance(cell, dict) else 0

# pick list of proteins
if PROTEIN_LOOP_LIST is not None:
    loop_proteins = [p for p in PROTEIN_LOOP_LIST if p in Y_sub.columns]
    missing = set(PROTEIN_LOOP_LIST) - set(loop_proteins)
    if missing:
        print("Warning: missing proteins ignored:", sorted(list(missing))[:10], "..." if len(missing)>10 else "")
else:
    loop_proteins = Y_sub.notna().sum().sort_values(ascending=False).head(int(PROTEIN_LOOP_TOP_K)).index.tolist()

print("Proteins to run:", loop_proteins)

batch_rows = []

for p in loop_proteins:
    idx = Y_sub[p].notna()
    Xp = X_sub.loc[idx]
    Yp = Y_sub.loc[idx]

    if Xp.shape[0] < 3:
        print(f"Skipping {p}: too few samples after NaN filtering.")
        continue

    print(f"Running feature_selection for {p} on {Xp.shape[0]} samples / {Xp.shape[1]} genes...")
    fs_p = sf.feature_selection(
        X=Xp,
        y=Yp,
        protein_name=p,
        percentile=BATCH_PERCENTILE,
        n_samples=None
    )

    cnts_p = fs_p.applymap(_count_dict_features)
    sumrow = {
        "protein": p,
        "anova_mean": float(cnts_p["anova"].mean()),
        "pearson_mean": float(cnts_p["pearson"].mean()),
        "spearman_mean": float(cnts_p["spearman"].mean()),
        "n_LOO": int(cnts_p.shape[0]),
        "n_genes": int(Xp.shape[1]),
        "n_samples": int(Xp.shape[0]),
    }
    batch_rows.append(sumrow)

    OUT_DIR = DATA_DIR
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    fs_out = OUT_DIR / f"{p}_feature_selection_subset.joblib"
    dump(fs_p, fs_out)

    cnts_csv = OUT_DIR / f"{p}_feature_selection_counts_subset.csv"
    cnts_p.to_csv(cnts_csv)

    print("Saved:", fs_out.name, "and", cnts_csv.name)

    if RUN_NAIVE_IN_LOOP:
        print(f"Running naive_cluster_feature_selection for {p}...")
        naive_p = sf.naive_cluster_feature_selection(X=Xp, y=Yp, protein_name=p)
        naive_counts = naive_p['pearson'].map(lambda lst: len(lst) if isinstance(lst, list) else 0)
        naive_csv = OUT_DIR / f"{p}_naive_feature_selection_counts_subset.csv"
        naive_counts.to_csv(naive_csv)
        dump(naive_p, OUT_DIR / f"{p}_naive_feature_selection_subset.joblib")
        print("Saved:", naive_csv.name)

batch_summary = pd.DataFrame(batch_rows).set_index("protein") if batch_rows else pd.DataFrame()
display(batch_summary)


Proteins to run: ['A2M']
Running feature_selection for A2M on 150 samples / 1000 genes...
Saved: A2M_feature_selection_subset.joblib and A2M_feature_selection_counts_subset.csv


C:\Users\hp\AppData\Local\Temp\ipykernel_20992\2252686482.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cnts_p = fs_p.applymap(_count_dict_features)


,anova_mean,pearson_mean,spearman_mean,n_LOO,n_genes,n_samples
protein,,,,,,
A2M,50.0,0.0,0.0,150,1000,150
